In [1]:
# Cell 0 | 导入依赖、设置折内MPNN运行路径和测试折

import json
import os
import pickle
import random
import sys

from pathlib import Path
from typing import Union

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import (
    GroupShuffleSplit,
    train_test_split,
)
from sklearn.preprocessing import StandardScaler

from torch.utils.data import (
    DataLoader,
    Dataset,
    Subset,
)
from torch_geometric.data import Batch
from torch_geometric.nn import (
    NNConv,
    global_mean_pool,
)


def find_project_root(
    start_path: Union[str, Path] = ".",
) -> Path:
    """向上查找同时包含data和notebooks的项目根目录。"""
    current_path = Path(start_path).resolve()

    for _ in range(20):
        if (
            (current_path / "data").exists()
            and (current_path / "notebooks").exists()
        ):
            return current_path

        if current_path.parent == current_path:
            break

        current_path = current_path.parent

    raise FileNotFoundError(
        "未找到项目根目录。请确认Notebook位于：\n"
        r"D:\jupyter\IL_ML_Project\notebooks"
    )


def seed_everything(seed: int) -> None:
    """固定Python、NumPy和PyTorch随机种子。"""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


ROOT = find_project_root(Path.cwd())

EXPERIMENT_ROOT = (
    ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
)

GNN_DIR = (
    EXPERIMENT_ROOT
    / "features"
    / "mpnn_graph"
)

REPRESENTATION_DIR = (
    EXPERIMENT_ROOT
    / "outputs"
    / "representation_compare"
)

CV_SPLIT_DIR = (
    EXPERIMENT_ROOT
    / "outputs"
    / "model_training_3models_cv"
    / "cv_splits"
)

FOLDWISE_OUTPUT_ROOT = (
    EXPERIMENT_ROOT
    / "outputs"
    / "foldwise_mpnn_cv_features"
)

CATION_GRAPHS_PATH = (
    GNN_DIR
    / "cation_graphs.pt"
)

ANION_GRAPHS_PATH = (
    GNN_DIR
    / "anion_graphs.pt"
)

GRAPH_MANIFEST_PATH = (
    GNN_DIR
    / "graph_manifest.csv"
)

CV_INDICES_PATH = (
    CV_SPLIT_DIR
    / "global_cv_indices_eta_10point_holdout.pkl"
)

CV_INPUT_MANIFEST_PATH = (
    CV_SPLIT_DIR
    / "cv_input_row_manifest_eta.csv"
)

CV_SPLIT_MANIFEST_PATH = (
    CV_SPLIT_DIR
    / "cv_split_manifest_eta_5fold.json"
)

SPLIT_ORDER = [
    "Data_split",
    "IL_split",
    "Cation_split",
    "Anion_split",
]

GROUP_COLUMN_BY_SPLIT = {
    "Data_split": None,
    "IL_split": "IL_key",
    "Cation_split": "cation_smiles",
    "Anion_split": "anion_smiles",
}

# 第一轮只测试Data_split的Fold 1
RUN_SPLIT_NAME = "Data_split"
RUN_FOLD = 1

BASE_SEED = 42
INTERNAL_VALIDATION_FRACTION = 0.20

FOLD_SEED = (
    BASE_SEED
    + SPLIT_ORDER.index(RUN_SPLIT_NAME) * 100
    + RUN_FOLD
)

RUN_OUTPUT_DIR = (
    FOLDWISE_OUTPUT_ROOT
    / RUN_SPLIT_NAME
    / f"Fold_{RUN_FOLD}"
)

required_input_files = [
    CATION_GRAPHS_PATH,
    ANION_GRAPHS_PATH,
    GRAPH_MANIFEST_PATH,
    CV_INDICES_PATH,
    CV_INPUT_MANIFEST_PATH,
    CV_SPLIT_MANIFEST_PATH,
]

missing_input_files = [
    path
    for path in required_input_files
    if not path.exists()
]

if missing_input_files:
    raise FileNotFoundError(
        "缺少以下输入文件：\n"
        + "\n".join(
            str(path)
            for path in missing_input_files
        )
    )

if RUN_SPLIT_NAME not in SPLIT_ORDER:
    raise ValueError(
        f"RUN_SPLIT_NAME错误：{RUN_SPLIT_NAME}"
    )

if RUN_FOLD not in [1, 2, 3, 4, 5]:
    raise ValueError(
        f"RUN_FOLD必须为1至5，当前为：{RUN_FOLD}"
    )

seed_everything(FOLD_SEED)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

PIN_MEMORY = device.type == "cuda"

print("python                   =", sys.executable)
print("device                   =", device)
print("ROOT                     =", ROOT)
print("EXPERIMENT_ROOT          =", EXPERIMENT_ROOT)
print("RUN_SPLIT_NAME           =", RUN_SPLIT_NAME)
print("RUN_FOLD                 =", RUN_FOLD)
print("FOLD_SEED                =", FOLD_SEED)
print("RUN_OUTPUT_DIR           =", RUN_OUTPUT_DIR)
print("external 10 points read  =", False)

python                   = D:\conda\envs\rdkit-env\python.exe
device                   = cpu
ROOT                     = D:\jupyter\IL_ML_Project
EXPERIMENT_ROOT          = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout
RUN_SPLIT_NAME           = Data_split
RUN_FOLD                 = 1
FOLD_SEED                = 43
RUN_OUTPUT_DIR           = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\foldwise_mpnn_cv_features\Data_split\Fold_1
external 10 points read  = False


In [2]:
# Cell 1 | 读取分子图、正式5折索引并核对外层训练测试集


def read_csv_robust(path: Path) -> pd.DataFrame:
    """依次尝试常见编码读取CSV。"""
    last_error = None

    for encoding in [
        "utf-8-sig",
        "utf-8",
        "gb18030",
    ]:
        try:
            return pd.read_csv(
                path,
                encoding=encoding,
            )

        except UnicodeDecodeError as error:
            last_error = error

    raise last_error


def load_torch_object(path: Path):
    """兼容不同PyTorch版本读取对象。"""
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

    except TypeError:
        return torch.load(
            path,
            map_location="cpu",
        )


graph_manifest = read_csv_robust(
    GRAPH_MANIFEST_PATH
)

cv_input_rows = read_csv_robust(
    CV_INPUT_MANIFEST_PATH
)

with open(
    CV_SPLIT_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as file:
    cv_split_manifest = json.load(file)

with open(
    CV_INDICES_PATH,
    "rb",
) as file:
    cv_indices_all = pickle.load(file)

all_cation_graphs = load_torch_object(
    CATION_GRAPHS_PATH
)

all_anion_graphs = load_torch_object(
    ANION_GRAPHS_PATH
)

for dataframe in [
    graph_manifest,
    cv_input_rows,
]:
    dataframe.columns = (
        dataframe.columns
        .astype(str)
        .str.strip()
    )

required_columns = [
    "source_row_idx",
    "IL_key",
    "cation_smiles",
    "anion_smiles",
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]

missing_graph_columns = [
    column
    for column in (
        required_columns
        + [
            "graph_valid",
            "graph_list_idx",
        ]
    )
    if column not in graph_manifest.columns
]

if missing_graph_columns:
    raise ValueError(
        "graph_manifest.csv缺少字段："
        f"{missing_graph_columns}"
    )

missing_cv_columns = [
    column
    for column in required_columns
    if column not in cv_input_rows.columns
]

if missing_cv_columns:
    raise ValueError(
        "cv_input_row_manifest_eta.csv缺少字段："
        f"{missing_cv_columns}"
    )

graph_manifest["graph_valid"] = pd.to_numeric(
    graph_manifest["graph_valid"],
    errors="raise",
).astype(int)

graph_manifest["graph_list_idx"] = pd.to_numeric(
    graph_manifest["graph_list_idx"],
    errors="raise",
).astype(int)

if not graph_manifest["graph_valid"].eq(1).all():
    raise RuntimeError(
        "当前正式数据中存在未成功构图的样本，"
        "不能直接使用现有交叉验证索引。"
    )

model_rows = (
    graph_manifest
    .sort_values("graph_list_idx")
    .reset_index(drop=True)
    .copy()
)

expected_graph_indices = np.arange(
    len(model_rows),
    dtype=int,
)

if not np.array_equal(
    model_rows["graph_list_idx"].to_numpy(dtype=int),
    expected_graph_indices,
):
    raise RuntimeError(
        "graph_list_idx不连续或顺序错误。"
    )

if not (
    len(model_rows)
    == len(cv_input_rows)
    == len(all_cation_graphs)
    == len(all_anion_graphs)
):
    raise RuntimeError(
        "数据、CV行清单、阳离子图和阴离子图数量不一致：\n"
        f"model_rows={len(model_rows)}\n"
        f"cv_input_rows={len(cv_input_rows)}\n"
        f"cation_graphs={len(all_cation_graphs)}\n"
        f"anion_graphs={len(all_anion_graphs)}"
    )

for column in [
    "source_row_idx",
    "IL_key",
    "cation_smiles",
    "anion_smiles",
]:
    if not np.array_equal(
        model_rows[column]
        .astype(str)
        .to_numpy(),
        cv_input_rows[column]
        .astype(str)
        .to_numpy(),
    ):
        raise RuntimeError(
            f"{column}在分子图清单与CV行清单中顺序不一致。"
        )

for column in [
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]:
    graph_values = pd.to_numeric(
        model_rows[column],
        errors="raise",
    ).to_numpy(dtype=float)

    cv_values = pd.to_numeric(
        cv_input_rows[column],
        errors="raise",
    ).to_numpy(dtype=float)

    if not np.allclose(
        graph_values,
        cv_values,
        rtol=0.0,
        atol=1e-12,
    ):
        raise RuntimeError(
            f"{column}在分子图清单与CV行清单中不一致。"
        )

if cv_split_manifest.get("n_splits") != 5:
    raise RuntimeError(
        "cv_split_manifest中的n_splits不是5。"
    )

if cv_split_manifest.get(
    "external_holdout_used_for_split"
) is not False:
    raise RuntimeError(
        "外部10点被标记为参与正式划分。"
    )

if cv_split_manifest.get(
    "split_order"
) != SPLIT_ORDER:
    raise RuntimeError(
        "四种划分顺序与当前设置不一致。"
    )

if int(
    cv_split_manifest["counts"]["model_rows"]
) != len(model_rows):
    raise RuntimeError(
        "cv_split_manifest中的样本数与当前数据不一致。"
    )

if "eta" not in cv_indices_all:
    raise KeyError(
        "正式交叉验证索引中不存在eta。"
    )

eta_indices = cv_indices_all["eta"]

if set(eta_indices.keys()) != set(SPLIT_ORDER):
    raise KeyError(
        "正式交叉验证索引中的四种划分不完整。"
    )

fold_payload = (
    eta_indices[RUN_SPLIT_NAME][RUN_FOLD - 1]
)

if not {"train", "test"}.issubset(
    fold_payload.keys()
):
    raise KeyError(
        "当前折索引缺少train或test。"
    )

outer_train_idx = np.asarray(
    fold_payload["train"],
    dtype=int,
)

outer_test_idx = np.asarray(
    fold_payload["test"],
    dtype=int,
)

n_samples = len(model_rows)

for index_name, index_values in [
    ("outer_train_idx", outer_train_idx),
    ("outer_test_idx", outer_test_idx),
]:
    if (
        index_values.min() < 0
        or index_values.max() >= n_samples
    ):
        raise IndexError(
            f"{index_name}存在越界索引。"
        )

if len(
    np.intersect1d(
        outer_train_idx,
        outer_test_idx,
    )
) != 0:
    raise RuntimeError(
        "外层训练集和测试集存在行号重叠。"
    )

outer_union = np.sort(
    np.concatenate(
        [
            outer_train_idx,
            outer_test_idx,
        ]
    )
)

if not np.array_equal(
    outer_union,
    np.arange(n_samples, dtype=int),
):
    raise RuntimeError(
        "当前外层训练集和测试集没有完整覆盖16216条数据。"
    )

group_column = GROUP_COLUMN_BY_SPLIT[
    RUN_SPLIT_NAME
]

outer_group_overlap = None

if group_column is not None:
    outer_train_groups = set(
        model_rows.iloc[
            outer_train_idx
        ][group_column].astype(str)
    )

    outer_test_groups = set(
        model_rows.iloc[
            outer_test_idx
        ][group_column].astype(str)
    )

    outer_group_overlap = (
        outer_train_groups
        & outer_test_groups
    )

    if outer_group_overlap:
        raise RuntimeError(
            f"{RUN_SPLIT_NAME}外层训练集和测试集"
            f"存在重复{group_column}。"
        )

env_vars_raw = (
    model_rows[
        [
            "T_rep",
            "P_rep",
        ]
    ]
    .apply(
        pd.to_numeric,
        errors="raise",
    )
    .to_numpy(dtype=float)
)

targets_raw = pd.to_numeric(
    model_rows["lg_eta"],
    errors="raise",
).to_numpy(dtype=float)

cation_graphs = [
    all_cation_graphs[index]
    for index in model_rows[
        "graph_list_idx"
    ].to_numpy(dtype=int)
]

anion_graphs = [
    all_anion_graphs[index]
    for index in model_rows[
        "graph_list_idx"
    ].to_numpy(dtype=int)
]

print("外层正式折读取和核对通过 ✓")
print("all rows                =", n_samples)
print("split                   =", RUN_SPLIT_NAME)
print("fold                    =", RUN_FOLD)
print("outer train rows        =", len(outer_train_idx))
print("outer test rows         =", len(outer_test_idx))
print(
    "outer train fraction    =",
    len(outer_train_idx) / n_samples,
)
print(
    "outer test fraction     =",
    len(outer_test_idx) / n_samples,
)
print(
    "row overlap             =",
    len(
        np.intersect1d(
            outer_train_idx,
            outer_test_idx,
        )
    ),
)
print(
    "group column            =",
    group_column,
)
print(
    "group overlap           =",
    (
        "not applicable"
        if group_column is None
        else len(outer_group_overlap)
    ),
)
print("unique ILs              =", model_rows["IL_key"].nunique())
print(
    "unique cations         =",
    model_rows["cation_smiles"].nunique(),
)
print(
    "unique anions          =",
    model_rows["anion_smiles"].nunique(),
)

外层正式折读取和核对通过 ✓
all rows                = 16216
split                   = Data_split
fold                    = 1
outer train rows        = 12972
outer test rows         = 3244
outer train fraction    = 0.7999506660088801
outer test fraction     = 0.20004933399111988
row overlap             = 0
group column            = None
group overlap           = not applicable
unique ILs              = 1295
unique cations         = 583
unique anions          = 202


In [3]:
# Cell 2 | 在外层训练折内部建立MPNN训练集、验证集和标准化器

if RUN_SPLIT_NAME == "Data_split":
    (
        mpnn_internal_train_idx,
        mpnn_internal_val_idx,
    ) = train_test_split(
        outer_train_idx,
        test_size=INTERNAL_VALIDATION_FRACTION,
        random_state=FOLD_SEED,
        shuffle=True,
    )

else:
    group_column = GROUP_COLUMN_BY_SPLIT[
        RUN_SPLIT_NAME
    ]

    outer_train_groups = (
        model_rows.iloc[
            outer_train_idx
        ][group_column]
        .astype(str)
        .to_numpy()
    )

    internal_splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=INTERNAL_VALIDATION_FRACTION,
        random_state=FOLD_SEED,
    )

    (
        internal_train_relative_idx,
        internal_val_relative_idx,
    ) = next(
        internal_splitter.split(
            X=np.zeros(
                len(outer_train_idx),
                dtype=np.float32,
            ),
            groups=outer_train_groups,
        )
    )

    mpnn_internal_train_idx = outer_train_idx[
        internal_train_relative_idx
    ]

    mpnn_internal_val_idx = outer_train_idx[
        internal_val_relative_idx
    ]

mpnn_internal_train_idx = np.asarray(
    mpnn_internal_train_idx,
    dtype=int,
)

mpnn_internal_val_idx = np.asarray(
    mpnn_internal_val_idx,
    dtype=int,
)

if len(
    np.intersect1d(
        mpnn_internal_train_idx,
        mpnn_internal_val_idx,
    )
) != 0:
    raise RuntimeError(
        "MPNN内部训练集和验证集存在行号重叠。"
    )

if len(
    np.intersect1d(
        mpnn_internal_train_idx,
        outer_test_idx,
    )
) != 0:
    raise RuntimeError(
        "MPNN内部训练集与外层测试集存在重叠。"
    )

if len(
    np.intersect1d(
        mpnn_internal_val_idx,
        outer_test_idx,
    )
) != 0:
    raise RuntimeError(
        "MPNN内部验证集与外层测试集存在重叠。"
    )

internal_union = np.sort(
    np.concatenate(
        [
            mpnn_internal_train_idx,
            mpnn_internal_val_idx,
        ]
    )
)

if not np.array_equal(
    internal_union,
    np.sort(outer_train_idx),
):
    raise RuntimeError(
        "MPNN内部训练集和验证集没有完整覆盖外层训练折。"
    )

internal_group_overlap = None

if RUN_SPLIT_NAME != "Data_split":
    group_column = GROUP_COLUMN_BY_SPLIT[
        RUN_SPLIT_NAME
    ]

    internal_train_groups = set(
        model_rows.iloc[
            mpnn_internal_train_idx
        ][group_column].astype(str)
    )

    internal_val_groups = set(
        model_rows.iloc[
            mpnn_internal_val_idx
        ][group_column].astype(str)
    )

    internal_group_overlap = (
        internal_train_groups
        & internal_val_groups
    )

    if internal_group_overlap:
        raise RuntimeError(
            "MPNN内部训练集和验证集存在重复分组："
            f"{group_column}"
        )

# 只使用MPNN内部训练部分拟合T/P标准化器
scaler_X = StandardScaler()

scaler_X.fit(
    env_vars_raw[
        mpnn_internal_train_idx
    ]
)

# 对全部样本应用同一个训练折标准化器；
# 外层测试折只进行transform，不参与fit
env_vars_scaled = scaler_X.transform(
    env_vars_raw
).astype(np.float32)

# 只使用MPNN内部训练部分拟合目标值标准化器
scaler_y = StandardScaler()

scaler_y.fit(
    targets_raw[
        mpnn_internal_train_idx
    ].reshape(-1, 1)
)

# 外层测试标签不转换、不进入MPNN训练数据
targets_scaled = np.full(
    n_samples,
    np.nan,
    dtype=np.float32,
)

targets_scaled[
    outer_train_idx
] = (
    scaler_y.transform(
        targets_raw[
            outer_train_idx
        ].reshape(-1, 1)
    )
    .reshape(-1)
    .astype(np.float32)
)

if not np.isfinite(
    env_vars_scaled
).all():
    raise RuntimeError(
        "标准化后的T/P中存在NaN或无穷值。"
    )

if not np.isfinite(
    targets_scaled[
        outer_train_idx
    ]
).all():
    raise RuntimeError(
        "外层训练折标准化目标值中存在NaN或无穷值。"
    )

if not np.isnan(
    targets_scaled[
        outer_test_idx
    ]
).all():
    raise RuntimeError(
        "外层测试标签被意外转换或写入训练数组。"
    )

print("MPNN折内训练/验证划分完成 ✓")
print("outer train rows        =", len(outer_train_idx))
print("outer test rows         =", len(outer_test_idx))
print(
    "MPNN internal train    =",
    len(mpnn_internal_train_idx),
)
print(
    "MPNN internal val      =",
    len(mpnn_internal_val_idx),
)
print(
    "internal train/all     =",
    len(mpnn_internal_train_idx) / n_samples,
)
print(
    "internal val/all       =",
    len(mpnn_internal_val_idx) / n_samples,
)
print(
    "internal train/outer   =",
    (
        len(mpnn_internal_train_idx)
        / len(outer_train_idx)
    ),
)
print(
    "internal val/outer     =",
    (
        len(mpnn_internal_val_idx)
        / len(outer_train_idx)
    ),
)
print(
    "train-val row overlap  =",
    len(
        np.intersect1d(
            mpnn_internal_train_idx,
            mpnn_internal_val_idx,
        )
    ),
)
print(
    "train-test row overlap =",
    len(
        np.intersect1d(
            mpnn_internal_train_idx,
            outer_test_idx,
        )
    ),
)
print(
    "val-test row overlap   =",
    len(
        np.intersect1d(
            mpnn_internal_val_idx,
            outer_test_idx,
        )
    ),
)
print(
    "internal group overlap =",
    (
        "not applicable"
        if RUN_SPLIT_NAME == "Data_split"
        else len(internal_group_overlap)
    ),
)
print(
    "T/P scaler fitted rows =",
    scaler_X.n_samples_seen_,
)
print(
    "y scaler fitted rows   =",
    scaler_y.n_samples_seen_,
)
print(
    "outer test y is NaN    =",
    np.isnan(
        targets_scaled[
            outer_test_idx
        ]
    ).all(),
)

MPNN折内训练/验证划分完成 ✓
outer train rows        = 12972
outer test rows         = 3244
MPNN internal train    = 10377
MPNN internal val      = 2595
internal train/all     = 0.6399235323137642
internal val/all       = 0.16002713369511593
internal train/outer   = 0.7999537465309898
internal val/outer     = 0.20004625346901017
train-val row overlap  = 0
train-test row overlap = 0
val-test row overlap   = 0
internal group overlap = not applicable
T/P scaler fitted rows = 10377
y scaler fitted rows   = 10377
outer test y is NaN    = True


In [4]:
# Cell 3 | 建立折内MPNN数据集和DataLoader

class ILDataset(Dataset):
    """离子液体阳离子图、阴离子图、环境条件和目标值数据集。"""

    def __init__(
        self,
        cation_graph_list,
        anion_graph_list,
        environment_array,
        target_array,
    ):
        if not (
            len(cation_graph_list)
            == len(anion_graph_list)
            == len(environment_array)
            == len(target_array)
        ):
            raise ValueError(
                "阳离子图、阴离子图、环境变量和目标值数量不一致。"
            )

        self.cation_graph_list = cation_graph_list
        self.anion_graph_list = anion_graph_list

        self.environment_tensor = torch.tensor(
            environment_array,
            dtype=torch.float32,
        )

        self.target_tensor = (
            torch.tensor(
                target_array,
                dtype=torch.float32,
            )
            .reshape(-1, 1)
        )

    def __len__(self):
        return len(self.target_tensor)

    def __getitem__(self, index):
        return (
            self.cation_graph_list[index],
            self.anion_graph_list[index],
            self.environment_tensor[index],
            self.target_tensor[index],
        )


def custom_collate(batch):
    """将一个批次中的阳、阴离子PyG图分别合并。"""

    cation_batch = Batch.from_data_list(
        [
            item[0]
            for item in batch
        ]
    )

    anion_batch = Batch.from_data_list(
        [
            item[1]
            for item in batch
        ]
    )

    environment_batch = torch.stack(
        [
            item[2]
            for item in batch
        ]
    )

    target_batch = torch.stack(
        [
            item[3]
            for item in batch
        ]
    )

    return (
        cation_batch,
        anion_batch,
        environment_batch,
        target_batch,
    )


dataset = ILDataset(
    cation_graph_list=cation_graphs,
    anion_graph_list=anion_graphs,
    environment_array=env_vars_scaled,
    target_array=targets_scaled,
)

mpnn_train_dataset = Subset(
    dataset,
    mpnn_internal_train_idx.tolist(),
)

mpnn_val_dataset = Subset(
    dataset,
    mpnn_internal_val_idx.tolist(),
)

outer_train_feature_dataset = Subset(
    dataset,
    outer_train_idx.tolist(),
)

outer_test_feature_dataset = Subset(
    dataset,
    outer_test_idx.tolist(),
)

BATCH_SIZE = 128
NUM_WORKERS = 0

train_generator = (
    torch.Generator()
    .manual_seed(FOLD_SEED)
)

mpnn_train_loader = DataLoader(
    mpnn_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=custom_collate,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    generator=train_generator,
    drop_last=False,
)

mpnn_train_eval_loader = DataLoader(
    mpnn_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=custom_collate,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

mpnn_val_loader = DataLoader(
    mpnn_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=custom_collate,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

outer_train_feature_loader = DataLoader(
    outer_train_feature_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=custom_collate,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

outer_test_feature_loader = DataLoader(
    outer_test_feature_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=custom_collate,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

# 检查内部训练和验证目标均为有限值
if not torch.isfinite(
    dataset.target_tensor[
        mpnn_internal_train_idx
    ]
).all():
    raise RuntimeError(
        "MPNN内部训练集中存在NaN或无穷目标值。"
    )

if not torch.isfinite(
    dataset.target_tensor[
        mpnn_internal_val_idx
    ]
).all():
    raise RuntimeError(
        "MPNN内部验证集中存在NaN或无穷目标值。"
    )

# 外层测试标签必须继续保持NaN
if not torch.isnan(
    dataset.target_tensor[
        outer_test_idx
    ]
).all():
    raise RuntimeError(
        "外层测试标签被意外写入MPNN数据集。"
    )

# 使用不打乱顺序的训练评价Loader进行批次结构检查
(
    check_cation_batch,
    check_anion_batch,
    check_environment_batch,
    check_target_batch,
) = next(
    iter(mpnn_train_eval_loader)
)

if check_cation_batch.x.shape[1] != 7:
    raise RuntimeError(
        "阳离子图原子特征维度不是7。"
    )

if check_anion_batch.x.shape[1] != 7:
    raise RuntimeError(
        "阴离子图原子特征维度不是7。"
    )

if check_cation_batch.edge_attr.shape[1] != 4:
    raise RuntimeError(
        "阳离子图化学键特征维度不是4。"
    )

if check_anion_batch.edge_attr.shape[1] != 4:
    raise RuntimeError(
        "阴离子图化学键特征维度不是4。"
    )

if check_environment_batch.shape[1] != 2:
    raise RuntimeError(
        "环境变量维度不是2。"
    )

if check_target_batch.shape[1] != 1:
    raise RuntimeError(
        "MPNN目标值批次维度不是1。"
    )

print("折内MPNN数据集和DataLoader建立完成 ✓")
print("dataset rows              =", len(dataset))
print("MPNN train rows           =", len(mpnn_train_dataset))
print("MPNN validation rows      =", len(mpnn_val_dataset))
print(
    "outer train feature rows =",
    len(outer_train_feature_dataset),
)
print(
    "outer test feature rows  =",
    len(outer_test_feature_dataset),
)
print("batch size                =", BATCH_SIZE)
print(
    "first batch graphs       =",
    check_cation_batch.num_graphs,
)
print(
    "cation node feature dim  =",
    check_cation_batch.x.shape[1],
)
print(
    "anion node feature dim   =",
    check_anion_batch.x.shape[1],
)
print(
    "cation edge feature dim  =",
    check_cation_batch.edge_attr.shape[1],
)
print(
    "anion edge feature dim   =",
    check_anion_batch.edge_attr.shape[1],
)
print(
    "environment batch shape =",
    tuple(check_environment_batch.shape),
)
print(
    "target batch shape      =",
    tuple(check_target_batch.shape),
)
print(
    "outer test target NaN   =",
    torch.isnan(
        dataset.target_tensor[
            outer_test_idx
        ]
    ).all().item(),
)

折内MPNN数据集和DataLoader建立完成 ✓
dataset rows              = 16216
MPNN train rows           = 10377
MPNN validation rows      = 2595
outer train feature rows = 12972
outer test feature rows  = 3244
batch size                = 128
first batch graphs       = 128
cation node feature dim  = 7
anion node feature dim   = 7
cation edge feature dim  = 4
anion edge feature dim   = 4
environment batch shape = (128, 2)
target batch shape      = (128, 1)
outer test target NaN   = True


In [5]:
# Cell 4 | 定义双分支MPNN编码器和黏度监督训练模型

class MPNNEncoder(nn.Module):
    """将单个离子分子图编码为64维图表征。"""

    def __init__(
        self,
        node_dim: int = 7,
        edge_dim: int = 4,
        hidden_dim: int = 64,
    ):
        super().__init__()

        self.node_dim = node_dim
        self.edge_dim = edge_dim
        self.hidden_dim = hidden_dim

        self.lin0 = nn.Linear(
            node_dim,
            hidden_dim,
        )

        edge_network = nn.Sequential(
            nn.Linear(
                edge_dim,
                32,
            ),
            nn.ReLU(),
            nn.Linear(
                32,
                hidden_dim * hidden_dim,
            ),
        )

        self.conv = NNConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            nn=edge_network,
            aggr="mean",
        )

        self.lin1 = nn.Linear(
            hidden_dim,
            hidden_dim,
        )

    def forward(self, graph_batch):
        node_state = self.lin0(
            graph_batch.x.float()
        )

        node_state = torch.relu(
            self.conv(
                node_state,
                graph_batch.edge_index,
                graph_batch.edge_attr.float(),
            )
        )

        node_state = self.lin1(
            node_state
        )

        graph_embedding = global_mean_pool(
            node_state,
            graph_batch.batch,
        )

        return graph_embedding


class ILPredictor(nn.Module):
    """
    双分支MPNN监督训练模型。

    阳离子表征64维，阴离子表征64维，
    再与标准化T/P两维拼接形成130维特征。
    """

    def __init__(self):
        super().__init__()

        self.cation_encoder = MPNNEncoder(
            node_dim=7,
            edge_dim=4,
            hidden_dim=64,
        )

        self.anion_encoder = MPNNEncoder(
            node_dim=7,
            edge_dim=4,
            hidden_dim=64,
        )

        self.regression_head = nn.Sequential(
            nn.Linear(
                130,
                64,
            ),
            nn.ReLU(),
            nn.Linear(
                64,
                32,
            ),
            nn.ReLU(),
            nn.Linear(
                32,
                1,
            ),
        )

    def extract_features(
        self,
        cation_batch,
        anion_batch,
        environment_batch,
    ):
        """提取130维中间特征，不经过最终回归头。"""

        cation_embedding = self.cation_encoder(
            cation_batch
        )

        anion_embedding = self.anion_encoder(
            anion_batch
        )

        features = torch.cat(
            [
                cation_embedding,
                anion_embedding,
                environment_batch,
            ],
            dim=1,
        )

        return features

    def forward(
        self,
        cation_batch,
        anion_batch,
        environment_batch,
    ):
        features = self.extract_features(
            cation_batch,
            anion_batch,
            environment_batch,
        )

        prediction = self.regression_head(
            features
        )

        return prediction


# 每一个外层折都重新初始化一个新的MPNN
seed_everything(FOLD_SEED)

model = ILPredictor().to(
    device
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

# 用一个内部训练批次检查网络输入和输出维度
model.eval()

with torch.no_grad():
    (
        check_cation_batch,
        check_anion_batch,
        check_environment_batch,
        _,
    ) = next(
        iter(mpnn_train_eval_loader)
    )

    check_cation_batch = check_cation_batch.to(
        device
    )

    check_anion_batch = check_anion_batch.to(
        device
    )

    check_environment_batch = (
        check_environment_batch.to(
            device
        )
    )

    check_features = model.extract_features(
        check_cation_batch,
        check_anion_batch,
        check_environment_batch,
    )

    check_prediction = model(
        check_cation_batch,
        check_anion_batch,
        check_environment_batch,
    )

if check_features.shape != (
    check_environment_batch.shape[0],
    130,
):
    raise RuntimeError(
        "MPNN中间特征维度错误："
        f"{tuple(check_features.shape)}"
    )

if check_prediction.shape != (
    check_environment_batch.shape[0],
    1,
):
    raise RuntimeError(
        "MPNN预测输出维度错误："
        f"{tuple(check_prediction.shape)}"
    )

print("双分支MPNN模型定义和维度检查完成 ✓")
print("split                    =", RUN_SPLIT_NAME)
print("fold                     =", RUN_FOLD)
print("fold seed                =", FOLD_SEED)
print("device                   =", device)
print("cation embedding dim     =", 64)
print("anion embedding dim      =", 64)
print("T/P feature dim          =", 2)
print("descriptor dim           =", 130)
print("total parameters         =", total_parameters)
print("trainable parameters     =", trainable_parameters)
print(
    "check feature shape     =",
    tuple(check_features.shape),
)
print(
    "check prediction shape  =",
    tuple(check_prediction.shape),
)

双分支MPNN模型定义和维度检查完成 ✓
split                    = Data_split
fold                     = 1
fold seed                = 43
device                   = cpu
cation embedding dim     = 64
anion embedding dim      = 64
T/P feature dim          = 2
descriptor dim           = 130
total parameters         = 298817
trainable parameters     = 298817
check feature shape     = (128, 130)
check prediction shape  = (128, 1)


In [6]:
# Cell 5 | 单阶段折内训练MPNN并保存最佳模型与训练记录

import time


# 当前折的输出文件
RUN_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRAINING_HISTORY_PATH = (
    RUN_OUTPUT_DIR
    / "mpnn_training_history.csv"
)

BEST_STATE_PATH = (
    RUN_OUTPUT_DIR
    / "mpnn_best_state.pt"
)

TRAINING_SUMMARY_PATH = (
    RUN_OUTPUT_DIR
    / "mpnn_training_summary.json"
)


# 每次运行本Cell都重新初始化模型，避免沿用上一次训练后的参数
seed_everything(FOLD_SEED)

model = ILPredictor().to(
    device
)


# 每次运行本Cell都重新建立带固定随机种子的训练Loader
train_generator = (
    torch.Generator()
    .manual_seed(FOLD_SEED)
)

mpnn_train_loader = DataLoader(
    mpnn_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=custom_collate,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    generator=train_generator,
    drop_last=False,
)


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5,
)

criterion = nn.MSELoss()

scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=8,
        min_lr=1e-7,
    )
)


MAX_EPOCHS = 150
EARLY_STOP_PATIENCE = 20
MIN_DELTA = 1e-6

best_val_loss = float("inf")
best_epoch = 0
best_state_dict = None
no_improve_count = 0

training_history = []

training_start_time = time.perf_counter()

print(
    "开始折内MPNN监督训练："
    f"{RUN_SPLIT_NAME} - Fold {RUN_FOLD}"
)

print(
    "MPNN internal train rows =",
    len(mpnn_train_dataset),
)

print(
    "MPNN internal val rows   =",
    len(mpnn_val_dataset),
)

print(
    f"MAX_EPOCHS={MAX_EPOCHS}, "
    f"EarlyStopping patience={EARLY_STOP_PATIENCE}"
)

print(
    "外层测试标签参与训练      =",
    False,
)


for epoch in range(
    1,
    MAX_EPOCHS + 1,
):
    # =========================
    # 训练阶段
    # =========================
    model.train()

    total_train_loss = 0.0
    train_sample_count = 0

    for (
        cation_batch,
        anion_batch,
        environment_batch,
        target_batch,
    ) in mpnn_train_loader:

        cation_batch = cation_batch.to(
            device
        )

        anion_batch = anion_batch.to(
            device
        )

        environment_batch = (
            environment_batch.to(
                device
            )
        )

        target_batch = target_batch.to(
            device
        )

        if not torch.isfinite(
            target_batch
        ).all():
            raise RuntimeError(
                "MPNN内部训练批次中存在NaN或无穷目标值。"
            )

        optimizer.zero_grad()

        prediction = model(
            cation_batch,
            anion_batch,
            environment_batch,
        )

        loss = criterion(
            prediction,
            target_batch,
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Epoch {epoch}训练损失出现NaN或无穷值。"
            )

        loss.backward()

        optimizer.step()

        current_batch_size = (
            target_batch.shape[0]
        )

        total_train_loss += (
            loss.item()
            * current_batch_size
        )

        train_sample_count += (
            current_batch_size
        )

    if train_sample_count != len(
        mpnn_train_dataset
    ):
        raise RuntimeError(
            "当前Epoch实际训练样本数"
            "与MPNN内部训练集行数不一致。"
        )

    average_train_loss = (
        total_train_loss
        / train_sample_count
    )

    # =========================
    # 内部验证阶段
    # =========================
    model.eval()

    total_val_loss = 0.0
    val_sample_count = 0

    with torch.no_grad():
        for (
            cation_batch,
            anion_batch,
            environment_batch,
            target_batch,
        ) in mpnn_val_loader:

            cation_batch = (
                cation_batch.to(
                    device
                )
            )

            anion_batch = (
                anion_batch.to(
                    device
                )
            )

            environment_batch = (
                environment_batch.to(
                    device
                )
            )

            target_batch = (
                target_batch.to(
                    device
                )
            )

            if not torch.isfinite(
                target_batch
            ).all():
                raise RuntimeError(
                    "MPNN内部验证批次中存在NaN或无穷目标值。"
                )

            prediction = model(
                cation_batch,
                anion_batch,
                environment_batch,
            )

            val_loss = criterion(
                prediction,
                target_batch,
            )

            if not torch.isfinite(
                val_loss
            ):
                raise RuntimeError(
                    f"Epoch {epoch}验证损失出现NaN或无穷值。"
                )

            current_batch_size = (
                target_batch.shape[0]
            )

            total_val_loss += (
                val_loss.item()
                * current_batch_size
            )

            val_sample_count += (
                current_batch_size
            )

    if val_sample_count != len(
        mpnn_val_dataset
    ):
        raise RuntimeError(
            "当前Epoch实际验证样本数"
            "与MPNN内部验证集行数不一致。"
        )

    average_val_loss = (
        total_val_loss
        / val_sample_count
    )

    scheduler.step(
        average_val_loss
    )

    current_learning_rate = float(
        optimizer.param_groups[0]["lr"]
    )

    improved = (
        average_val_loss
        < best_val_loss - MIN_DELTA
    )

    if improved:
        best_val_loss = float(
            average_val_loss
        )

        best_epoch = int(
            epoch
        )

        best_state_dict = {
            key: value
            .detach()
            .cpu()
            .clone()

            for key, value
            in model.state_dict().items()
        }

        no_improve_count = 0

        torch.save(
            {
                "property": "viscosity",
                "target_col": "lg_eta",
                "representation_method": "MPNN",
                "split_name": RUN_SPLIT_NAME,
                "fold": int(RUN_FOLD),
                "fold_seed": int(FOLD_SEED),
                "model_class": "ILPredictor",
                "encoder_class": "MPNNEncoder",
                "node_feature_dim": 7,
                "edge_feature_dim": 4,
                "cation_embedding_dim": 64,
                "anion_embedding_dim": 64,
                "environment_dim": 2,
                "descriptor_dim": 130,
                "best_epoch": int(best_epoch),
                "best_val_loss": float(
                    best_val_loss
                ),
                "model_state_dict": (
                    best_state_dict
                ),
            },
            BEST_STATE_PATH,
        )

    else:
        no_improve_count += 1

    training_history.append(
        {
            "split_name": RUN_SPLIT_NAME,
            "fold": int(RUN_FOLD),
            "fold_seed": int(FOLD_SEED),
            "epoch": int(epoch),
            "train_loss": float(
                average_train_loss
            ),
            "val_loss": float(
                average_val_loss
            ),
            "best_val_loss": float(
                best_val_loss
            ),
            "best_epoch": int(
                best_epoch
            ),
            "learning_rate": float(
                current_learning_rate
            ),
            "improved": int(
                improved
            ),
            "no_improve_count": int(
                no_improve_count
            ),
        }
    )

    if (
        epoch == 1
        or epoch % 10 == 0
        or improved
        or epoch == MAX_EPOCHS
    ):
        print(
            f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
            f"Train Loss: {average_train_loss:.6f} | "
            f"Val Loss: {average_val_loss:.6f} | "
            f"Best Val: {best_val_loss:.6f} "
            f"@ Epoch {best_epoch:03d} | "
            f"LR: {current_learning_rate:.8f}"
        )

    if (
        no_improve_count
        >= EARLY_STOP_PATIENCE
    ):
        print(
            "Early stopping：连续"
            f"{EARLY_STOP_PATIENCE}轮"
            "内部验证损失未明显下降。"
        )

        print(
            "停止轮次 =",
            epoch,
        )

        break


training_end_time = time.perf_counter()

training_seconds = float(
    training_end_time
    - training_start_time
)

epochs_ran = len(
    training_history
)


if best_state_dict is None:
    raise RuntimeError(
        "训练过程中没有保存到最佳模型。"
    )


# 恢复内部验证集上表现最好的MPNN参数
model.load_state_dict(
    best_state_dict,
    strict=True,
)

model = model.to(
    device
)

model.eval()


training_history_df = pd.DataFrame(
    training_history
)

training_history_df.to_csv(
    TRAINING_HISTORY_PATH,
    index=False,
    encoding="utf-8-sig",
)


training_summary = {
    "property": "viscosity",
    "target_col": "lg_eta",
    "representation_method": "MPNN",
    "split_name": RUN_SPLIT_NAME,
    "fold": int(RUN_FOLD),
    "fold_seed": int(FOLD_SEED),

    "all_rows": int(
        n_samples
    ),

    "outer_train_rows": int(
        len(outer_train_idx)
    ),

    "outer_test_rows": int(
        len(outer_test_idx)
    ),

    "mpnn_internal_train_rows": int(
        len(mpnn_internal_train_idx)
    ),

    "mpnn_internal_validation_rows": int(
        len(mpnn_internal_val_idx)
    ),

    "external_holdout_used": False,

    "outer_test_used_for_training": False,

    "standardization": {
        "tp_scaler_fitted_rows": int(
            scaler_X.n_samples_seen_
        ),
        "target_scaler_fitted_rows": int(
            scaler_y.n_samples_seen_
        ),
    },

    "training": {
        "optimizer": "Adam",
        "initial_learning_rate": 0.001,
        "weight_decay": 1e-5,
        "loss": "MSELoss",
        "scheduler": "ReduceLROnPlateau",
        "scheduler_factor": 0.5,
        "scheduler_patience": 8,
        "maximum_epochs": int(
            MAX_EPOCHS
        ),
        "early_stopping_patience": int(
            EARLY_STOP_PATIENCE
        ),
        "minimum_delta": float(
            MIN_DELTA
        ),
        "epochs_ran": int(
            epochs_ran
        ),
        "best_epoch": int(
            best_epoch
        ),
        "best_val_loss_standardized": float(
            best_val_loss
        ),
        "training_seconds": float(
            training_seconds
        ),
    },

    "architecture": {
        "node_feature_dim": 7,
        "edge_feature_dim": 4,
        "cation_embedding_dim": 64,
        "anion_embedding_dim": 64,
        "environment_dim": 2,
        "descriptor_dim": 130,
        "trainable_parameters": int(
            sum(
                parameter.numel()
                for parameter
                in model.parameters()
                if parameter.requires_grad
            )
        ),
    },

    "output_files": {
        "training_history": str(
            TRAINING_HISTORY_PATH
        ),
        "best_state": str(
            BEST_STATE_PATH
        ),
    },
}


with open(
    TRAINING_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        training_summary,
        file,
        ensure_ascii=False,
        indent=2,
    )


print()
print("折内MPNN监督训练完成 ✓")
print("split                   =", RUN_SPLIT_NAME)
print("fold                    =", RUN_FOLD)
print("epochs actually ran     =", epochs_ran)
print("best epoch              =", best_epoch)
print("best validation loss    =", best_val_loss)
print(
    "training minutes        =",
    training_seconds / 60.0,
)
print("outer test used         =", False)
print("saved:", TRAINING_HISTORY_PATH)
print("saved:", BEST_STATE_PATH)
print("saved:", TRAINING_SUMMARY_PATH)

开始折内MPNN监督训练：Data_split - Fold 1
MPNN internal train rows = 10377
MPNN internal val rows   = 2595
MAX_EPOCHS=150, EarlyStopping patience=20
外层测试标签参与训练      = False
Epoch 001/150 | Train Loss: 0.749935 | Val Loss: 0.655299 | Best Val: 0.655299 @ Epoch 001 | LR: 0.00100000
Epoch 002/150 | Train Loss: 0.509765 | Val Loss: 0.446137 | Best Val: 0.446137 @ Epoch 002 | LR: 0.00100000
Epoch 003/150 | Train Loss: 0.418018 | Val Loss: 0.379196 | Best Val: 0.379196 @ Epoch 003 | LR: 0.00100000
Epoch 005/150 | Train Loss: 0.323382 | Val Loss: 0.324319 | Best Val: 0.324319 @ Epoch 005 | LR: 0.00100000
Epoch 006/150 | Train Loss: 0.287421 | Val Loss: 0.266830 | Best Val: 0.266830 @ Epoch 006 | LR: 0.00100000
Epoch 007/150 | Train Loss: 0.278867 | Val Loss: 0.244119 | Best Val: 0.244119 @ Epoch 007 | LR: 0.00100000
Epoch 009/150 | Train Loss: 0.240455 | Val Loss: 0.230418 | Best Val: 0.230418 @ Epoch 009 | LR: 0.00100000
Epoch 010/150 | Train Loss: 0.199474 | Val Loss: 0.276824 | Best Val: 0.230418 @

In [7]:
# Cell 6 | 计算折内MPNN指标并提取外层训练集和测试集特征

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


def calculate_regression_metrics(
    y_true,
    y_pred,
):
    """计算lg_eta原始尺度上的回归指标。"""

    y_true = np.asarray(
        y_true,
        dtype=float,
    ).reshape(-1)

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    ).reshape(-1)

    r2 = r2_score(
        y_true,
        y_pred,
    )

    rmse = float(
        np.sqrt(
            mean_squared_error(
                y_true,
                y_pred,
            )
        )
    )

    mae = mean_absolute_error(
        y_true,
        y_pred,
    )

    if (
        np.std(y_true) == 0
        or np.std(y_pred) == 0
    ):
        pearson_r = np.nan

    else:
        pearson_r = float(
            np.corrcoef(
                y_true,
                y_pred,
            )[0, 1]
        )

    return {
        "R2": float(r2),
        "RMSE": float(rmse),
        "MAE": float(mae),
        "Pearson_r": pearson_r,
    }


def collect_original_scale_predictions(
    data_loader,
):
    """
    收集MPNN内部训练集或验证集的真实值和预测值，
    并从标准化空间还原到lg_eta原始尺度。
    """

    model.eval()

    true_scaled_batches = []
    pred_scaled_batches = []

    with torch.no_grad():
        for (
            cation_batch,
            anion_batch,
            environment_batch,
            target_batch,
        ) in data_loader:

            cation_batch = cation_batch.to(
                device
            )

            anion_batch = anion_batch.to(
                device
            )

            environment_batch = (
                environment_batch.to(
                    device
                )
            )

            target_batch = target_batch.to(
                device
            )

            prediction_batch = model(
                cation_batch,
                anion_batch,
                environment_batch,
            )

            if not torch.isfinite(
                prediction_batch
            ).all():
                raise RuntimeError(
                    "MPNN预测结果中存在NaN或无穷值。"
                )

            true_scaled_batches.append(
                target_batch
                .detach()
                .cpu()
                .numpy()
            )

            pred_scaled_batches.append(
                prediction_batch
                .detach()
                .cpu()
                .numpy()
            )

    true_scaled = np.vstack(
        true_scaled_batches
    )

    pred_scaled = np.vstack(
        pred_scaled_batches
    )

    true_original = (
        scaler_y
        .inverse_transform(
            true_scaled
        )
        .reshape(-1)
    )

    pred_original = (
        scaler_y
        .inverse_transform(
            pred_scaled
        )
        .reshape(-1)
    )

    return (
        true_original,
        pred_original,
    )


def extract_feature_matrix(
    data_loader,
):
    """使用当前折最佳MPNN提取130维特征。"""

    model.eval()

    feature_batches = []

    with torch.no_grad():
        for (
            cation_batch,
            anion_batch,
            environment_batch,
            _,
        ) in data_loader:

            cation_batch = cation_batch.to(
                device
            )

            anion_batch = anion_batch.to(
                device
            )

            environment_batch = (
                environment_batch.to(
                    device
                )
            )

            batch_features = (
                model.extract_features(
                    cation_batch,
                    anion_batch,
                    environment_batch,
                )
            )

            if not torch.isfinite(
                batch_features
            ).all():
                raise RuntimeError(
                    "提取的MPNN特征中存在NaN或无穷值。"
                )

            feature_batches.append(
                batch_features
                .detach()
                .cpu()
                .numpy()
            )

    return np.vstack(
        feature_batches
    ).astype(np.float32)


# 计算MPNN内部训练集和验证集预测
internal_train_true, internal_train_pred = (
    collect_original_scale_predictions(
        mpnn_train_eval_loader
    )
)

internal_val_true, internal_val_pred = (
    collect_original_scale_predictions(
        mpnn_val_loader
    )
)


# 检查还原后的真实值是否与原始lg_eta严格对齐
max_internal_train_alignment_error = float(
    np.max(
        np.abs(
            internal_train_true
            - targets_raw[
                mpnn_internal_train_idx
            ]
        )
    )
)

max_internal_val_alignment_error = float(
    np.max(
        np.abs(
            internal_val_true
            - targets_raw[
                mpnn_internal_val_idx
            ]
        )
    )
)

if (
    max_internal_train_alignment_error
    > 1e-5
):
    raise RuntimeError(
        "MPNN内部训练集目标值还原或行顺序错误。"
    )

if (
    max_internal_val_alignment_error
    > 1e-5
):
    raise RuntimeError(
        "MPNN内部验证集目标值还原或行顺序错误。"
    )


internal_train_metrics = (
    calculate_regression_metrics(
        internal_train_true,
        internal_train_pred,
    )
)

internal_val_metrics = (
    calculate_regression_metrics(
        internal_val_true,
        internal_val_pred,
    )
)

internal_metrics_df = pd.DataFrame(
    [
        {
            "split_name": RUN_SPLIT_NAME,
            "fold": int(RUN_FOLD),
            "evaluation_subset": (
                "mpnn_internal_train"
            ),
            "n_samples": int(
                len(internal_train_true)
            ),
            **internal_train_metrics,
        },
        {
            "split_name": RUN_SPLIT_NAME,
            "fold": int(RUN_FOLD),
            "evaluation_subset": (
                "mpnn_internal_validation"
            ),
            "n_samples": int(
                len(internal_val_true)
            ),
            **internal_val_metrics,
        },
    ]
)


# 提取整个外层训练折和外层测试折的130维特征
outer_train_features = extract_feature_matrix(
    outer_train_feature_loader
)

outer_test_features = extract_feature_matrix(
    outer_test_feature_loader
)


expected_outer_train_shape = (
    len(outer_train_idx),
    130,
)

expected_outer_test_shape = (
    len(outer_test_idx),
    130,
)

if (
    outer_train_features.shape
    != expected_outer_train_shape
):
    raise RuntimeError(
        "外层训练特征矩阵形状错误："
        f"{outer_train_features.shape}，"
        f"预期为{expected_outer_train_shape}。"
    )

if (
    outer_test_features.shape
    != expected_outer_test_shape
):
    raise RuntimeError(
        "外层测试特征矩阵形状错误："
        f"{outer_test_features.shape}，"
        f"预期为{expected_outer_test_shape}。"
    )

if not np.isfinite(
    outer_train_features
).all():
    raise RuntimeError(
        "外层训练特征中存在NaN或无穷值。"
    )

if not np.isfinite(
    outer_test_features
).all():
    raise RuntimeError(
        "外层测试特征中存在NaN或无穷值。"
    )


# 检查最后两维是否为当前折标准化后的T/P
max_outer_train_environment_error = float(
    np.max(
        np.abs(
            outer_train_features[
                :,
                128:130,
            ]
            - env_vars_scaled[
                outer_train_idx
            ]
        )
    )
)

max_outer_test_environment_error = float(
    np.max(
        np.abs(
            outer_test_features[
                :,
                128:130,
            ]
            - env_vars_scaled[
                outer_test_idx
            ]
        )
    )
)

if (
    max_outer_train_environment_error
    > 1e-6
):
    raise RuntimeError(
        "外层训练特征最后两维与标准化T/P不一致。"
    )

if (
    max_outer_test_environment_error
    > 1e-6
):
    raise RuntimeError(
        "外层测试特征最后两维与标准化T/P不一致。"
    )


# 构建内部训练和验证预测明细，供下一Cell保存
internal_train_predictions_df = (
    model_rows.iloc[
        mpnn_internal_train_idx
    ][
        [
            "source_row_idx",
            "graph_list_idx",
            "IL_key",
            "cation_smiles",
            "anion_smiles",
            "T_rep",
            "P_rep",
            "eta",
            "lg_eta",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

internal_train_predictions_df[
    "lg_eta_true"
] = internal_train_true

internal_train_predictions_df[
    "lg_eta_pred"
] = internal_train_pred

internal_train_predictions_df[
    "residual_pred_minus_true"
] = (
    internal_train_pred
    - internal_train_true
)

internal_train_predictions_df[
    "absolute_error"
] = np.abs(
    internal_train_pred
    - internal_train_true
)


internal_val_predictions_df = (
    model_rows.iloc[
        mpnn_internal_val_idx
    ][
        [
            "source_row_idx",
            "graph_list_idx",
            "IL_key",
            "cation_smiles",
            "anion_smiles",
            "T_rep",
            "P_rep",
            "eta",
            "lg_eta",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

internal_val_predictions_df[
    "lg_eta_true"
] = internal_val_true

internal_val_predictions_df[
    "lg_eta_pred"
] = internal_val_pred

internal_val_predictions_df[
    "residual_pred_minus_true"
] = (
    internal_val_pred
    - internal_val_true
)

internal_val_predictions_df[
    "absolute_error"
] = np.abs(
    internal_val_pred
    - internal_val_true
)


print("折内MPNN指标和外层特征提取完成 ✓")
print()
print(
    internal_metrics_df.to_string(
        index=False
    )
)
print()
print(
    "outer train feature shape =",
    outer_train_features.shape,
)
print(
    "outer test feature shape  =",
    outer_test_features.shape,
)
print(
    "outer train feature dtype =",
    outer_train_features.dtype,
)
print(
    "outer test feature dtype  =",
    outer_test_features.dtype,
)
print(
    "train target align error  =",
    max_internal_train_alignment_error,
)
print(
    "val target align error    =",
    max_internal_val_alignment_error,
)
print(
    "train T/P feature error   =",
    max_outer_train_environment_error,
)
print(
    "test T/P feature error    =",
    max_outer_test_environment_error,
)
print(
    "outer test labels used    =",
    False,
)

折内MPNN指标和外层特征提取完成 ✓

split_name  fold        evaluation_subset  n_samples       R2     RMSE      MAE  Pearson_r
Data_split     1      mpnn_internal_train      10377 0.982494 0.109247 0.068355   0.991263
Data_split     1 mpnn_internal_validation       2595 0.972149 0.138247 0.082864   0.986020

outer train feature shape = (12972, 130)
outer test feature shape  = (3244, 130)
outer train feature dtype = float32
outer test feature dtype  = float32
train target align error  = 7.172314937520241e-07
val target align error    = 4.2509495479237103e-07
train T/P feature error   = 0.0
test T/P feature error    = 0.0
outer test labels used    = False


In [8]:
# Cell 7 | 保存当前外层折的MPNN特征、索引、清单、指标和标准化器

RUN_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# 当前折特征
OUTER_TRAIN_FEATURE_PATH = (
    RUN_OUTPUT_DIR
    / "X_outer_train_mpnn.npy"
)

OUTER_TEST_FEATURE_PATH = (
    RUN_OUTPUT_DIR
    / "X_outer_test_mpnn.npy"
)

# 当前折目标值，仅供下游模型训练和测试评价使用
OUTER_TRAIN_TARGET_PATH = (
    RUN_OUTPUT_DIR
    / "y_outer_train_lg_eta.npy"
)

OUTER_TEST_TARGET_PATH = (
    RUN_OUTPUT_DIR
    / "y_outer_test_lg_eta.npy"
)

# 当前折索引
FOLD_INDICES_PATH = (
    RUN_OUTPUT_DIR
    / "fold_indices.npz"
)

# 行顺序清单
OUTER_TRAIN_MANIFEST_PATH = (
    RUN_OUTPUT_DIR
    / "outer_train_manifest.csv"
)

OUTER_TEST_MANIFEST_PATH = (
    RUN_OUTPUT_DIR
    / "outer_test_manifest.csv"
)

# MPNN内部检查结果
INTERNAL_METRICS_PATH = (
    RUN_OUTPUT_DIR
    / "mpnn_internal_metrics.csv"
)

INTERNAL_TRAIN_PREDICTIONS_PATH = (
    RUN_OUTPUT_DIR
    / "mpnn_internal_train_predictions.csv"
)

INTERNAL_VAL_PREDICTIONS_PATH = (
    RUN_OUTPUT_DIR
    / "mpnn_internal_validation_predictions.csv"
)

# 当前折标准化器
TP_SCALER_PATH = (
    RUN_OUTPUT_DIR
    / "tp_scaler.joblib"
)

TARGET_SCALER_PATH = (
    RUN_OUTPUT_DIR
    / "target_scaler.joblib"
)

# 当前折总清单
FOLD_FEATURE_METADATA_PATH = (
    RUN_OUTPUT_DIR
    / "fold_feature_metadata.json"
)


# =========================
# 1. 保存特征矩阵
# =========================

np.save(
    OUTER_TRAIN_FEATURE_PATH,
    outer_train_features,
)

np.save(
    OUTER_TEST_FEATURE_PATH,
    outer_test_features,
)


# =========================
# 2. 保存训练和测试目标值
# =========================

outer_train_targets = (
    targets_raw[
        outer_train_idx
    ]
    .astype(np.float32)
)

outer_test_targets = (
    targets_raw[
        outer_test_idx
    ]
    .astype(np.float32)
)

if not np.isfinite(
    outer_train_targets
).all():
    raise RuntimeError(
        "外层训练目标值中存在NaN或无穷值。"
    )

if not np.isfinite(
    outer_test_targets
).all():
    raise RuntimeError(
        "外层测试目标值中存在NaN或无穷值。"
    )

np.save(
    OUTER_TRAIN_TARGET_PATH,
    outer_train_targets,
)

np.save(
    OUTER_TEST_TARGET_PATH,
    outer_test_targets,
)


# =========================
# 3. 保存全部索引
# =========================

np.savez_compressed(
    FOLD_INDICES_PATH,

    outer_train_idx=(
        outer_train_idx
        .astype(int)
    ),

    outer_test_idx=(
        outer_test_idx
        .astype(int)
    ),

    mpnn_internal_train_idx=(
        mpnn_internal_train_idx
        .astype(int)
    ),

    mpnn_internal_validation_idx=(
        mpnn_internal_val_idx
        .astype(int)
    ),
)


# =========================
# 4. 建立外层训练和测试行清单
# =========================

manifest_columns = [
    "source_row_idx",
    "graph_list_idx",
    "IL_key",
    "cation_smiles",
    "anion_smiles",
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]

outer_train_manifest = (
    model_rows.iloc[
        outer_train_idx
    ][manifest_columns]
    .reset_index(drop=True)
    .copy()
)

outer_train_manifest.insert(
    0,
    "fold_feature_row_idx",
    np.arange(
        len(outer_train_manifest),
        dtype=int,
    ),
)

outer_train_manifest.insert(
    0,
    "original_feature_row_idx",
    outer_train_idx.astype(int),
)

outer_train_manifest.insert(
    0,
    "subset",
    "outer_train",
)

outer_train_manifest.insert(
    0,
    "fold",
    int(RUN_FOLD),
)

outer_train_manifest.insert(
    0,
    "split_name",
    RUN_SPLIT_NAME,
)


outer_test_manifest = (
    model_rows.iloc[
        outer_test_idx
    ][manifest_columns]
    .reset_index(drop=True)
    .copy()
)

outer_test_manifest.insert(
    0,
    "fold_feature_row_idx",
    np.arange(
        len(outer_test_manifest),
        dtype=int,
    ),
)

outer_test_manifest.insert(
    0,
    "original_feature_row_idx",
    outer_test_idx.astype(int),
)

outer_test_manifest.insert(
    0,
    "subset",
    "outer_test",
)

outer_test_manifest.insert(
    0,
    "fold",
    int(RUN_FOLD),
)

outer_test_manifest.insert(
    0,
    "split_name",
    RUN_SPLIT_NAME,
)


# 核对清单中的目标值与保存的目标数组一致
if not np.allclose(
    outer_train_manifest[
        "lg_eta"
    ].to_numpy(dtype=float),
    outer_train_targets,
    rtol=0.0,
    atol=1e-6,
):
    raise RuntimeError(
        "外层训练清单与训练目标数组顺序不一致。"
    )

if not np.allclose(
    outer_test_manifest[
        "lg_eta"
    ].to_numpy(dtype=float),
    outer_test_targets,
    rtol=0.0,
    atol=1e-6,
):
    raise RuntimeError(
        "外层测试清单与测试目标数组顺序不一致。"
    )

outer_train_manifest.to_csv(
    OUTER_TRAIN_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

outer_test_manifest.to_csv(
    OUTER_TEST_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)


# =========================
# 5. 保存MPNN内部检查结果
# =========================

internal_metrics_df.to_csv(
    INTERNAL_METRICS_PATH,
    index=False,
    encoding="utf-8-sig",
)

internal_train_predictions_df.to_csv(
    INTERNAL_TRAIN_PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig",
)

internal_val_predictions_df.to_csv(
    INTERNAL_VAL_PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig",
)


# =========================
# 6. 保存当前折标准化器
# =========================

joblib.dump(
    scaler_X,
    TP_SCALER_PATH,
)

joblib.dump(
    scaler_y,
    TARGET_SCALER_PATH,
)


# =========================
# 7. 保存当前折metadata
# =========================

fold_feature_metadata = {
    "property": "viscosity",
    "target_col": "lg_eta",
    "representation_method": "MPNN",

    "split_name": RUN_SPLIT_NAME,
    "fold": int(RUN_FOLD),
    "fold_seed": int(FOLD_SEED),

    "external_holdout": {
        "n_points": 10,
        "used_in_current_fold": False,
    },

    "data_counts": {
        "all_rows": int(
            n_samples
        ),

        "outer_train_rows": int(
            len(outer_train_idx)
        ),

        "outer_test_rows": int(
            len(outer_test_idx)
        ),

        "mpnn_internal_train_rows": int(
            len(mpnn_internal_train_idx)
        ),

        "mpnn_internal_validation_rows": int(
            len(mpnn_internal_val_idx)
        ),
    },

    "leakage_control": {
        "outer_train_test_row_overlap": 0,

        "mpnn_train_outer_test_row_overlap": 0,

        "mpnn_validation_outer_test_row_overlap": 0,

        "outer_test_used_for_mpnn_training": False,

        "outer_test_used_for_early_stopping": False,

        "tp_scaler_fitted_only_on_mpnn_internal_train": True,

        "target_scaler_fitted_only_on_mpnn_internal_train": True,

        "outer_test_target_saved_for_evaluation_only": True,
    },

    "architecture": {
        "node_feature_dim": 7,
        "edge_feature_dim": 4,
        "cation_embedding_dim": 64,
        "anion_embedding_dim": 64,
        "environment_dim": 2,
        "descriptor_dim": 130,
        "trainable_parameters": 298817,
    },

    "training": {
        "maximum_epochs": int(
            MAX_EPOCHS
        ),

        "epochs_ran": int(
            epochs_ran
        ),

        "best_epoch": int(
            best_epoch
        ),

        "best_validation_loss_standardized": float(
            best_val_loss
        ),

        "training_seconds": float(
            training_seconds
        ),
    },

    "internal_metrics": (
        internal_metrics_df
        .to_dict(
            orient="records"
        )
    ),

    "feature_matrices": {
        "outer_train_shape": list(
            outer_train_features.shape
        ),

        "outer_test_shape": list(
            outer_test_features.shape
        ),

        "dtype": str(
            outer_train_features.dtype
        ),

        "outer_train_tp_alignment_error": float(
            max_outer_train_environment_error
        ),

        "outer_test_tp_alignment_error": float(
            max_outer_test_environment_error
        ),
    },

    "output_files": {
        "outer_train_features": str(
            OUTER_TRAIN_FEATURE_PATH
        ),

        "outer_test_features": str(
            OUTER_TEST_FEATURE_PATH
        ),

        "outer_train_targets": str(
            OUTER_TRAIN_TARGET_PATH
        ),

        "outer_test_targets": str(
            OUTER_TEST_TARGET_PATH
        ),

        "fold_indices": str(
            FOLD_INDICES_PATH
        ),

        "outer_train_manifest": str(
            OUTER_TRAIN_MANIFEST_PATH
        ),

        "outer_test_manifest": str(
            OUTER_TEST_MANIFEST_PATH
        ),

        "internal_metrics": str(
            INTERNAL_METRICS_PATH
        ),

        "internal_train_predictions": str(
            INTERNAL_TRAIN_PREDICTIONS_PATH
        ),

        "internal_validation_predictions": str(
            INTERNAL_VAL_PREDICTIONS_PATH
        ),

        "best_mpnn_state": str(
            BEST_STATE_PATH
        ),

        "training_history": str(
            TRAINING_HISTORY_PATH
        ),

        "training_summary": str(
            TRAINING_SUMMARY_PATH
        ),

        "tp_scaler": str(
            TP_SCALER_PATH
        ),

        "target_scaler": str(
            TARGET_SCALER_PATH
        ),
    },
}

with open(
    FOLD_FEATURE_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        fold_feature_metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )


print("Data_split - Fold 1特征和审计文件保存完成 ✓")
print("split                    =", RUN_SPLIT_NAME)
print("fold                     =", RUN_FOLD)
print(
    "outer train feature     =",
    outer_train_features.shape,
)
print(
    "outer test feature      =",
    outer_test_features.shape,
)
print(
    "outer train target      =",
    outer_train_targets.shape,
)
print(
    "outer test target       =",
    outer_test_targets.shape,
)
print(
    "outer test used in MPNN =",
    False,
)

print("\n输出文件完整路径：")

saved_paths = [
    OUTER_TRAIN_FEATURE_PATH,
    OUTER_TEST_FEATURE_PATH,
    OUTER_TRAIN_TARGET_PATH,
    OUTER_TEST_TARGET_PATH,
    FOLD_INDICES_PATH,
    OUTER_TRAIN_MANIFEST_PATH,
    OUTER_TEST_MANIFEST_PATH,
    INTERNAL_METRICS_PATH,
    INTERNAL_TRAIN_PREDICTIONS_PATH,
    INTERNAL_VAL_PREDICTIONS_PATH,
    TP_SCALER_PATH,
    TARGET_SCALER_PATH,
    FOLD_FEATURE_METADATA_PATH,
    BEST_STATE_PATH,
    TRAINING_HISTORY_PATH,
    TRAINING_SUMMARY_PATH,
]

for path in saved_paths:
    print(path)

Data_split - Fold 1特征和审计文件保存完成 ✓
split                    = Data_split
fold                     = 1
outer train feature     = (12972, 130)
outer test feature      = (3244, 130)
outer train target      = (12972,)
outer test target       = (3244,)
outer test used in MPNN = False

输出文件完整路径：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\foldwise_mpnn_cv_features\Data_split\Fold_1\X_outer_train_mpnn.npy
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\foldwise_mpnn_cv_features\Data_split\Fold_1\X_outer_test_mpnn.npy
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\foldwise_mpnn_cv_features\Data_split\Fold_1\y_outer_train_lg_eta.npy
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\foldwise_mpnn_cv_features\Data_split\Fold_1\y_outer_test_lg_eta.npy
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\foldwise_mpnn_cv_features\Data_split\Fold_1\fold_indices.npz


In [9]:
# Cell 8 | 从硬盘重新读取并审计Data_split-Fold 1全部输出

import math


FOLD_OUTPUT_AUDIT_PATH = (
    RUN_OUTPUT_DIR
    / "fold_output_audit.json"
)


def load_torch_checkpoint(path: Path):
    """兼容不同PyTorch版本读取checkpoint。"""
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

    except TypeError:
        return torch.load(
            path,
            map_location="cpu",
        )


def fitted_sample_count(scaler) -> int:
    """读取StandardScaler拟合样本数。"""
    sample_count = np.asarray(
        scaler.n_samples_seen_
    )

    return int(
        sample_count.max()
    )


expected_output_files = [
    OUTER_TRAIN_FEATURE_PATH,
    OUTER_TEST_FEATURE_PATH,
    OUTER_TRAIN_TARGET_PATH,
    OUTER_TEST_TARGET_PATH,
    FOLD_INDICES_PATH,
    OUTER_TRAIN_MANIFEST_PATH,
    OUTER_TEST_MANIFEST_PATH,
    INTERNAL_METRICS_PATH,
    INTERNAL_TRAIN_PREDICTIONS_PATH,
    INTERNAL_VAL_PREDICTIONS_PATH,
    TP_SCALER_PATH,
    TARGET_SCALER_PATH,
    FOLD_FEATURE_METADATA_PATH,
    BEST_STATE_PATH,
    TRAINING_HISTORY_PATH,
    TRAINING_SUMMARY_PATH,
]

missing_output_files = [
    path
    for path in expected_output_files
    if not path.exists()
]

if missing_output_files:
    raise FileNotFoundError(
        "当前折缺少以下输出文件：\n"
        + "\n".join(
            str(path)
            for path in missing_output_files
        )
    )


# =========================
# 1. 重新读取特征和目标
# =========================

train_feature_check = np.load(
    OUTER_TRAIN_FEATURE_PATH
)

test_feature_check = np.load(
    OUTER_TEST_FEATURE_PATH
)

train_target_check = np.load(
    OUTER_TRAIN_TARGET_PATH
)

test_target_check = np.load(
    OUTER_TEST_TARGET_PATH
)

if train_feature_check.shape != (
    len(outer_train_idx),
    130,
):
    raise RuntimeError(
        "硬盘中的外层训练特征形状错误："
        f"{train_feature_check.shape}"
    )

if test_feature_check.shape != (
    len(outer_test_idx),
    130,
):
    raise RuntimeError(
        "硬盘中的外层测试特征形状错误："
        f"{test_feature_check.shape}"
    )

if train_target_check.shape != (
    len(outer_train_idx),
):
    raise RuntimeError(
        "硬盘中的外层训练目标形状错误："
        f"{train_target_check.shape}"
    )

if test_target_check.shape != (
    len(outer_test_idx),
):
    raise RuntimeError(
        "硬盘中的外层测试目标形状错误："
        f"{test_target_check.shape}"
    )

for array_name, array_value in [
    (
        "train_feature_check",
        train_feature_check,
    ),
    (
        "test_feature_check",
        test_feature_check,
    ),
    (
        "train_target_check",
        train_target_check,
    ),
    (
        "test_target_check",
        test_target_check,
    ),
]:
    if not np.isfinite(
        array_value
    ).all():
        raise RuntimeError(
            f"{array_name}中存在NaN或无穷值。"
        )


# =========================
# 2. 重新读取索引
# =========================

with np.load(
    FOLD_INDICES_PATH
) as index_file:

    saved_outer_train_idx = (
        index_file[
            "outer_train_idx"
        ]
        .astype(int)
    )

    saved_outer_test_idx = (
        index_file[
            "outer_test_idx"
        ]
        .astype(int)
    )

    saved_mpnn_train_idx = (
        index_file[
            "mpnn_internal_train_idx"
        ]
        .astype(int)
    )

    saved_mpnn_val_idx = (
        index_file[
            "mpnn_internal_validation_idx"
        ]
        .astype(int)
    )

if not np.array_equal(
    saved_outer_train_idx,
    outer_train_idx,
):
    raise RuntimeError(
        "保存的外层训练索引与正式5折索引不一致。"
    )

if not np.array_equal(
    saved_outer_test_idx,
    outer_test_idx,
):
    raise RuntimeError(
        "保存的外层测试索引与正式5折索引不一致。"
    )

if not np.array_equal(
    saved_mpnn_train_idx,
    mpnn_internal_train_idx,
):
    raise RuntimeError(
        "保存的MPNN内部训练索引不一致。"
    )

if not np.array_equal(
    saved_mpnn_val_idx,
    mpnn_internal_val_idx,
):
    raise RuntimeError(
        "保存的MPNN内部验证索引不一致。"
    )

if np.intersect1d(
    saved_outer_train_idx,
    saved_outer_test_idx,
).size != 0:
    raise RuntimeError(
        "保存的外层训练和测试索引存在重叠。"
    )

if np.intersect1d(
    saved_mpnn_train_idx,
    saved_outer_test_idx,
).size != 0:
    raise RuntimeError(
        "保存的MPNN训练集与外层测试集存在重叠。"
    )

if np.intersect1d(
    saved_mpnn_val_idx,
    saved_outer_test_idx,
).size != 0:
    raise RuntimeError(
        "保存的MPNN验证集与外层测试集存在重叠。"
    )


# =========================
# 3. 重新读取行清单并核对顺序
# =========================

train_manifest_check = pd.read_csv(
    OUTER_TRAIN_MANIFEST_PATH,
    encoding="utf-8-sig",
)

test_manifest_check = pd.read_csv(
    OUTER_TEST_MANIFEST_PATH,
    encoding="utf-8-sig",
)

if len(train_manifest_check) != len(
    saved_outer_train_idx
):
    raise RuntimeError(
        "外层训练清单行数错误。"
    )

if len(test_manifest_check) != len(
    saved_outer_test_idx
):
    raise RuntimeError(
        "外层测试清单行数错误。"
    )

if not np.array_equal(
    train_manifest_check[
        "original_feature_row_idx"
    ].to_numpy(dtype=int),
    saved_outer_train_idx,
):
    raise RuntimeError(
        "外层训练清单行顺序与训练索引不一致。"
    )

if not np.array_equal(
    test_manifest_check[
        "original_feature_row_idx"
    ].to_numpy(dtype=int),
    saved_outer_test_idx,
):
    raise RuntimeError(
        "外层测试清单行顺序与测试索引不一致。"
    )

if not np.array_equal(
    train_manifest_check[
        "fold_feature_row_idx"
    ].to_numpy(dtype=int),
    np.arange(
        len(train_manifest_check),
        dtype=int,
    ),
):
    raise RuntimeError(
        "外层训练清单中的折内行号不连续。"
    )

if not np.array_equal(
    test_manifest_check[
        "fold_feature_row_idx"
    ].to_numpy(dtype=int),
    np.arange(
        len(test_manifest_check),
        dtype=int,
    ),
):
    raise RuntimeError(
        "外层测试清单中的折内行号不连续。"
    )

if not np.allclose(
    train_manifest_check[
        "lg_eta"
    ].to_numpy(dtype=float),
    train_target_check,
    rtol=0.0,
    atol=1e-6,
):
    raise RuntimeError(
        "外层训练清单与训练目标文件不一致。"
    )

if not np.allclose(
    test_manifest_check[
        "lg_eta"
    ].to_numpy(dtype=float),
    test_target_check,
    rtol=0.0,
    atol=1e-6,
):
    raise RuntimeError(
        "外层测试清单与测试目标文件不一致。"
    )


# =========================
# 4. 重新读取标准化器并核对T/P
# =========================

tp_scaler_check = joblib.load(
    TP_SCALER_PATH
)

target_scaler_check = joblib.load(
    TARGET_SCALER_PATH
)

if fitted_sample_count(
    tp_scaler_check
) != len(mpnn_internal_train_idx):
    raise RuntimeError(
        "T/P标准化器拟合样本数错误。"
    )

if fitted_sample_count(
    target_scaler_check
) != len(mpnn_internal_train_idx):
    raise RuntimeError(
        "目标值标准化器拟合样本数错误。"
    )

train_tp_recalculated = (
    tp_scaler_check
    .transform(
        train_manifest_check[
            [
                "T_rep",
                "P_rep",
            ]
        ].to_numpy(dtype=float)
    )
    .astype(np.float32)
)

test_tp_recalculated = (
    tp_scaler_check
    .transform(
        test_manifest_check[
            [
                "T_rep",
                "P_rep",
            ]
        ].to_numpy(dtype=float)
    )
    .astype(np.float32)
)

saved_train_tp_error = float(
    np.max(
        np.abs(
            train_feature_check[
                :,
                128:130,
            ]
            - train_tp_recalculated
        )
    )
)

saved_test_tp_error = float(
    np.max(
        np.abs(
            test_feature_check[
                :,
                128:130,
            ]
            - test_tp_recalculated
        )
    )
)

if saved_train_tp_error > 1e-6:
    raise RuntimeError(
        "硬盘训练特征最后两维与T/P标准化结果不一致。"
    )

if saved_test_tp_error > 1e-6:
    raise RuntimeError(
        "硬盘测试特征最后两维与T/P标准化结果不一致。"
    )


# =========================
# 5. 重新读取训练历史和指标
# =========================

history_check = pd.read_csv(
    TRAINING_HISTORY_PATH,
    encoding="utf-8-sig",
)

metrics_check = pd.read_csv(
    INTERNAL_METRICS_PATH,
    encoding="utf-8-sig",
)

if len(history_check) != epochs_ran:
    raise RuntimeError(
        "训练历史行数与实际训练轮数不一致。"
    )

history_best_row = history_check.loc[
    history_check[
        "val_loss"
    ].idxmin()
]

history_best_epoch = int(
    history_best_row["epoch"]
)

history_best_val_loss = float(
    history_best_row["val_loss"]
)

if history_best_epoch != best_epoch:
    raise RuntimeError(
        "训练历史中的最佳轮次不一致。"
    )

if not np.isclose(
    history_best_val_loss,
    best_val_loss,
    rtol=1e-7,
    atol=1e-9,
):
    raise RuntimeError(
        "训练历史中的最佳验证损失不一致。"
    )

required_metric_subsets = {
    "mpnn_internal_train",
    "mpnn_internal_validation",
}

actual_metric_subsets = set(
    metrics_check[
        "evaluation_subset"
    ].astype(str)
)

if actual_metric_subsets != required_metric_subsets:
    raise RuntimeError(
        "MPNN内部指标文件的数据集名称错误。"
    )

if not np.isfinite(
    metrics_check[
        [
            "R2",
            "RMSE",
            "MAE",
            "Pearson_r",
        ]
    ].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "MPNN内部指标中存在NaN或无穷值。"
    )


# =========================
# 6. 重新读取checkpoint和metadata
# =========================

checkpoint_check = load_torch_checkpoint(
    BEST_STATE_PATH
)

required_checkpoint_keys = [
    "property",
    "target_col",
    "split_name",
    "fold",
    "descriptor_dim",
    "best_epoch",
    "best_val_loss",
    "model_state_dict",
]

missing_checkpoint_keys = [
    key
    for key in required_checkpoint_keys
    if key not in checkpoint_check
]

if missing_checkpoint_keys:
    raise RuntimeError(
        "MPNN checkpoint缺少字段："
        f"{missing_checkpoint_keys}"
    )

if checkpoint_check[
    "split_name"
] != RUN_SPLIT_NAME:
    raise RuntimeError(
        "MPNN checkpoint中的划分名称错误。"
    )

if int(
    checkpoint_check["fold"]
) != RUN_FOLD:
    raise RuntimeError(
        "MPNN checkpoint中的折号错误。"
    )

if int(
    checkpoint_check["descriptor_dim"]
) != 130:
    raise RuntimeError(
        "MPNN checkpoint中的特征维度错误。"
    )

if int(
    checkpoint_check["best_epoch"]
) != best_epoch:
    raise RuntimeError(
        "MPNN checkpoint中的最佳轮次错误。"
    )

if not np.isclose(
    float(
        checkpoint_check[
            "best_val_loss"
        ]
    ),
    best_val_loss,
    rtol=1e-7,
    atol=1e-9,
):
    raise RuntimeError(
        "MPNN checkpoint中的最佳验证损失错误。"
    )

model_reload_check = ILPredictor()

model_reload_check.load_state_dict(
    checkpoint_check[
        "model_state_dict"
    ],
    strict=True,
)

with open(
    TRAINING_SUMMARY_PATH,
    "r",
    encoding="utf-8",
) as file:
    training_summary_check = json.load(file)

with open(
    FOLD_FEATURE_METADATA_PATH,
    "r",
    encoding="utf-8",
) as file:
    feature_metadata_check = json.load(file)

if training_summary_check[
    "outer_test_used_for_training"
] is not False:
    raise RuntimeError(
        "training summary错误地标记外层测试集参与训练。"
    )

leakage_check = feature_metadata_check[
    "leakage_control"
]

required_leakage_flags = {
    "outer_train_test_row_overlap": 0,
    "mpnn_train_outer_test_row_overlap": 0,
    "mpnn_validation_outer_test_row_overlap": 0,
    "outer_test_used_for_mpnn_training": False,
    "outer_test_used_for_early_stopping": False,
    "tp_scaler_fitted_only_on_mpnn_internal_train": True,
    "target_scaler_fitted_only_on_mpnn_internal_train": True,
    "outer_test_target_saved_for_evaluation_only": True,
}

for flag_name, expected_value in (
    required_leakage_flags.items()
):
    if leakage_check.get(
        flag_name
    ) != expected_value:
        raise RuntimeError(
            "metadata中的泄露控制字段错误："
            f"{flag_name}"
        )


# =========================
# 7. 保存最终审计结果
# =========================

fold_output_audit = {
    "audit_status": "passed",

    "property": "viscosity",
    "target_col": "lg_eta",

    "split_name": RUN_SPLIT_NAME,
    "fold": int(RUN_FOLD),
    "fold_seed": int(FOLD_SEED),

    "files_checked": int(
        len(expected_output_files)
    ),

    "data_counts": {
        "all_rows": int(n_samples),
        "outer_train_rows": int(
            len(saved_outer_train_idx)
        ),
        "outer_test_rows": int(
            len(saved_outer_test_idx)
        ),
        "mpnn_internal_train_rows": int(
            len(saved_mpnn_train_idx)
        ),
        "mpnn_internal_validation_rows": int(
            len(saved_mpnn_val_idx)
        ),
    },

    "feature_shapes": {
        "outer_train": list(
            train_feature_check.shape
        ),
        "outer_test": list(
            test_feature_check.shape
        ),
    },

    "target_shapes": {
        "outer_train": list(
            train_target_check.shape
        ),
        "outer_test": list(
            test_target_check.shape
        ),
    },

    "leakage_checks": {
        "outer_train_test_overlap": 0,
        "mpnn_train_outer_test_overlap": 0,
        "mpnn_validation_outer_test_overlap": 0,
        "outer_test_used_for_mpnn_training": False,
        "outer_test_used_for_early_stopping": False,
    },

    "standardization_checks": {
        "tp_scaler_fitted_rows": int(
            fitted_sample_count(
                tp_scaler_check
            )
        ),

        "target_scaler_fitted_rows": int(
            fitted_sample_count(
                target_scaler_check
            )
        ),

        "saved_outer_train_tp_error": float(
            saved_train_tp_error
        ),

        "saved_outer_test_tp_error": float(
            saved_test_tp_error
        ),
    },

    "training_checks": {
        "epochs_ran": int(
            len(history_check)
        ),

        "best_epoch": int(
            history_best_epoch
        ),

        "best_validation_loss_standardized": float(
            history_best_val_loss
        ),
    },

    "message": (
        "Data_split Fold 1折内监督MPNN输出、"
        "索引、标准化器、特征和训练记录"
        "全部检查通过。"
    ),
}

with open(
    FOLD_OUTPUT_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        fold_output_audit,
        file,
        ensure_ascii=False,
        indent=2,
    )


print("=" * 84)
print("Data_split - Fold 1全部输出最终审计通过 ✓")
print("=" * 84)

print("\n数据与特征：")
print(
    "outer train features     =",
    train_feature_check.shape,
)
print(
    "outer test features      =",
    test_feature_check.shape,
)
print(
    "outer train targets      =",
    train_target_check.shape,
)
print(
    "outer test targets       =",
    test_target_check.shape,
)

print("\n泄露控制：")
print(
    "outer train-test overlap =",
    0,
)
print(
    "MPNN train-test overlap  =",
    0,
)
print(
    "MPNN val-test overlap    =",
    0,
)
print(
    "outer test used in MPNN  =",
    False,
)
print(
    "outer test used in stop  =",
    False,
)

print("\n标准化与对齐：")
print(
    "T/P scaler fitted rows   =",
    fitted_sample_count(
        tp_scaler_check
    ),
)
print(
    "y scaler fitted rows     =",
    fitted_sample_count(
        target_scaler_check
    ),
)
print(
    "saved train T/P error    =",
    saved_train_tp_error,
)
print(
    "saved test T/P error     =",
    saved_test_tp_error,
)

print("\n训练记录：")
print(
    "epochs ran               =",
    len(history_check),
)
print(
    "best epoch               =",
    history_best_epoch,
)
print(
    "best validation loss     =",
    history_best_val_loss,
)

print("\n文件检查：")
print(
    "checked files            =",
    len(expected_output_files),
)
print(
    "missing files            =",
    0,
)
print(
    "audit status             =",
    "passed",
)
print(
    "saved audit              =",
    FOLD_OUTPUT_AUDIT_PATH,
)

Data_split - Fold 1全部输出最终审计通过 ✓

数据与特征：
outer train features     = (12972, 130)
outer test features      = (3244, 130)
outer train targets      = (12972,)
outer test targets       = (3244,)

泄露控制：
outer train-test overlap = 0
MPNN train-test overlap  = 0
MPNN val-test overlap    = 0
outer test used in MPNN  = False
outer test used in stop  = False

标准化与对齐：
T/P scaler fitted rows   = 10377
y scaler fitted rows     = 10377
saved train T/P error    = 0.0
saved test T/P error     = 0.0

训练记录：
epochs ran               = 150
best epoch               = 144
best validation loss     = 0.0280339269137982

文件检查：
checked files            = 16
missing files            = 0
audit status             = passed
saved audit              = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\foldwise_mpnn_cv_features\Data_split\Fold_1\fold_output_audit.json
